In [1]:
%%writefile dataloader_utils.py
import torch
import numpy as np
import random

def set_worker_seed(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    random.seed(worker_seed)
    np.random.seed(worker_seed)

def flatten_image(x):
    return x.view(-1)

Overwriting dataloader_utils.py


In [1]:
import torch
import torch.nn as nn
from torch.nn.utils import parameters_to_vector, vector_to_parameters
from torch.utils.data import DataLoader
from torchvision import datasets
import numpy as np
import random
from multiprocessing import cpu_count
from tqdm import trange
from torch import optim
import torch.nn.functional as F
from collections import defaultdict
import polars as pl
from torchsummary import summary
from lets_plot import *
LetsPlot.setup_html()
from torchvision.transforms import v2
from dataloader_utils import set_worker_seed, flatten_image

/Users/omar/Python/bgd/.venv/lib/python3.14/site-packages/lets_plot/plot/annotation.py:592: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
  .line(r'\(R\^2=\)@..r2..')\\
/Users/omar/Python/bgd/.venv/lib/python3.14/site-packages/lets_plot/plot/annotation.py:644: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
  .line(r'\(R\^2=\)@..r2..')\\


In [ ]:
def closure(x, y):
    out = model(x)
    loss = criterion(out, y)
    loss.backward()
    return loss

In [39]:

class MLP(nn.Module):
    def __init__(self, num_layers=3, in_dims=784):
        super().__init__()
        self.layers = nn.ModuleList()
        middle_layer = num_layers//2
        for layer in range(num_layers):
            div = 2 if layer != middle_layer else 1
            out_dims = in_dims//div
            self.layers.append(
                nn.Sequential(
                    nn.Linear(in_dims, out_dims),
                    nn.ReLU(inplace=True),
            ))
            in_dims = out_dims

        self.classify = nn.Linear(out_dims, 10)

    def forward(self, x: torch.Tensor):
        for layer in self.layers:
            x = layer(x)

        return self.classify(x)

model = MLP()



In [38]:
from typing import Callable, Iterable

import torch
import torch.nn as nn
from torch.nn.utils import parameters_to_vector, vector_to_parameters
from torch.optim import Optimizer


class BGD(Optimizer):
    r"""
    Implements BGD (Bouncing Gradient Descent).
    Args:
        params (iterable): iterable of parameters to optimize or dicts defining
            parameter groups
        lr (float): base learning rate (default: 0.1)
        beta (float): momentum factor (default: 0.9)
    Typical training procedure:
        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            def closure():
                loss = criterion(m(x), y)
                loss.backward()
                return loss.item()
            opt.step(closure)
    """

    def __init__(
        self, params: Iterable[torch.Tensor], lr: float = 0.3, beta: float = 0.9, weight_decay: float = 0.0
    ):
        if lr < 0.0:
            raise ValueError(f"Invalid learning rate: {lr}")
        if beta < 0.0:
            raise ValueError(f"Invalid beta value: {beta}")
        if weight_decay < 0.0:
            raise ValueError(f"Invalid weight_decay value: {weight_decay}")

        decay_params = []
        no_decay_params = []

        for p in params:
            if not p.requires_grad:
                continue
            # Exclude biases and 1D normalization parameters
            if p.ndim == 1:
                no_decay_params.append(p)
            else:
                decay_params.append(p)

        optim_groups = [
            {"params": decay_params, "weight_decay": weight_decay},
            {"params": no_decay_params, "weight_decay": 0.0}
        ]

        defaults = dict(lr=lr, beta=beta, weight_decay=weight_decay)
        super().__init__(optim_groups, defaults)  # exposes "self.param_groups" attribute

        self._params: list[nn.Parameter] = [
            p for p in self.param_groups[0]["params"] if p.requires_grad
        ]

        # Flatten entire model:
        with torch.no_grad():
            self.P = parameters_to_vector(self._params)
        self._prev_P: torch.Tensor = torch.empty_like(self.P)
        self._v: torch.Tensor = torch.zeros_like(self.P)
        self._G: torch.Tensor = torch.empty_like(self.P)

In [17]:
# -------------------------
# Config
# -------------------------
DEVICE = "mps"
SEED = 0
BS = 256
EPOCHS = 30
LR = 0.05
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    elif torch.mps.is_available():
        torch.mps.manual_seed(seed)


# class Act(nn.Module):
#     def __init__(self, dim: int):
#         super().__init__()



class MLP(nn.Module):
    def __init__(self, num_layers=3, in_dims=784):
        super().__init__()
        self.layers = nn.ModuleList()
        middle_layer = num_layers//2
        for layer in range(num_layers):
            div = 2 if layer != middle_layer else 1
            out_dims = in_dims//div
            self.layers.append(
                nn.Sequential(
                    nn.Linear(in_dims, out_dims),
                    nn.ReLU(inplace=True),
            ))
            in_dims = out_dims

        self.classify = nn.Linear(out_dims, 10)

    def forward(self, x: torch.Tensor):
        for layer in self.layers:
            x = layer(x)

        return self.classify(x)


def make_weight_decay_groups(model, weight_decay=0.0):
    decay_params = []
    no_decay_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.endswith("bias") or param.ndim == 1:
            no_decay_params.append(param)
        else:
            decay_params.append(param)

    return [
        {"params": decay_params, "weight_decay": weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0},
    ]


def train(model, opt, epochs, train_loader):
    for _ in trange(epochs):
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            loss = F.cross_entropy(model(x), y)
            opt.zero_grad()
            loss.backward()
            opt.step()


@torch.inference_mode()
def test(model, test_loader):
    model.eval()
    correct = 0
    total = 0

    for x, y in test_loader:
        x, y = x.to(DEVICE,), y.to(DEVICE,)
        logits = model(x)

        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    acc = correct / total
    print(f"Test Accuracy: {acc * 100:.2f}%")


def main():
    torch.mps.empty_cache()
    transform = v2.Compose([
        v2.PILToTensor(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize((0.2860,), (0.3530,)),  # common FashionMNIST mean/std
        v2.Lambda(flatten_image)  # lambda functions are NOT picklable, unless defined on the global scope
    ])

    train_ds = datasets.FashionMNIST(root="~/Python/datasets/FashionMNIST", train=True, download=False, transform=transform)
    test_ds = datasets.FashionMNIST(root="~/Python/datasets/FashionMNIST", train=False, download=False, transform=transform)
    set_seed(SEED)
    train_loader = DataLoader(train_ds,
                              batch_size=BS,
                              shuffle=True,
                              num_workers=NUM_WORKERS,  # torch pickles "init_fn" + dataset and all its transforms and sends serialized copy to each worker
                              persistent_workers=NUM_WORKERS>0,
                              drop_last=True,
                              worker_init_fn=set_worker_seed,
                              generator=torch.Generator().manual_seed(SEED))

    test_loader = DataLoader(test_ds, batch_size=BS, shuffle=False, num_workers=NUM_WORKERS, persistent_workers=False)

    model = MLP().to(DEVICE)
    run_logs = {}

    # The hook function must always accept these exact three arguments
    # @torch.no_grad()
    def hook_fn(module, in_layer, out_layer):
        out_detached = out_layer.detach()
        idx = module.idx
        cnt = out_detached.count_nonzero()
        run_logs[idx]["act"].append(out_detached.sum().div_(cnt.clamp_min(1)).item())
        run_logs[idx]["pct"].append(cnt.div_(out_detached.numel()).mul_(100.0).item())

    handles = []
    for i, layer in enumerate(model.layers):
        run_logs[i] = defaultdict(list)
        layer.idx = i
        handle = layer.register_forward_hook(hook_fn)  # We save the "handle" so we can remove the hook later if needed
        handles.append(handle)

    sgd = optim.SGD(make_weight_decay_groups(model, WEIGHT_DECAY), lr=LR)

    train(model, sgd, EPOCHS, train_loader)

    for handle in handles:
        handle.remove()

    test(model, test_loader)
    return run_logs


if __name__ == '__main__':
    data = main()


100%|██████████| 30/30 [01:23<00:00,  2.78s/it]


Test Accuracy: 88.59%


In [ ]:
df = pl.DataFrame([{"layer": k, **v} for k, v in data.items()]).explode("act", "pct")
df = df.with_columns(pl.row_index().over("layer"))

plot_df = (
    df.unpivot(
        index=["layer", "index"],
        on=["act", "pct"],
        variable_name="metric",
        value_name="value",
    )
    .with_columns(
        pl.when(pl.col("metric") == "act")
        .then(pl.lit("Mean non-zero activation"))
        .otherwise(pl.lit("Active units (%)"))
        .alias("metric")
    )
)

In [ ]:
p = (ggplot(plot_df)
 + geom_line(aes(x="index", y="value", color=as_discrete("layer")), size=0.7)
+ geom_smooth(aes(x="index", y="value"), se=True, method='loess')
 + facet_wrap("metric", ncol=1, scales="free_y")
 + labs(
     title="Layer activation dynamics during training",
     subtitle="Mean active ReLU output and percent of active units by layer",
     x="Training step",
     y="",
     color="Layer",
 )
 + ggsize(900, 650)
)

ggsave(p, f"lr={LR}.png", path="plots/")

In [ ]:
import torch

In [ ]:
x = torch.randn(128, 784)

In [ ]:
%timeit torch.einsum("ni,nj->nij", x, x)

In [ ]:
%timeit x.unsqueeze(-1) * x.unsqueeze(1)

In [ ]:
%timeit x.unsqueeze(-1) @ x.unsqueeze(1)